# Bài 2: Ứng dụng FastText Pretrained để phân loại cảm xúc

**Mục tiêu:** Sử dụng mô hình FastText đã được huấn luyện sẵn cho tiếng Việt để tạo vector đặc trưng cho câu, sau đó sử dụng MLP Classifier để phân loại.
**Luồng xử lý:** Load Pretrained -> Sentence Embedding (Trung bình cộng Word Vectors) -> MLP Classifier -> Đánh giá.

In [ ]:
import numpy as np
import requests
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from gensim.models import KeyedVectors

CONFIG = {
    "pretrained_path": "wiki.vi.vec", 
    "hidden_layers": (100, 50),
    "max_iter": 300
}

In [ ]:
def load_data(url):
    """Tải dữ liệu từ GitHub."""
    response = requests.get(url)
    lines = response.text.strip().split('\n')
    texts, labels = [], []
    for line in lines:
        parts = line.split(' ', 1)
        if len(parts) == 2:
            labels.append(1 if parts[0] == '__label__1' else 0)
            texts.append(parts[1])
    return texts, labels

train_url = "https://raw.githubusercontent.com/congnghia0609/ntc-scv/master/data/train.txt"
test_url = "https://raw.githubusercontent.com/congnghia0609/ntc-scv/master/data/test.txt"

X_train_raw, y_train = load_data(train_url)
X_test_raw, y_test = load_data(test_url)

In [ ]:
def get_sentence_vector(text, model):
    """Tính toán vector đại diện cho cả câu."""
    words = text.lower().split()
    vectors = [model[w] for w in words if w in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [ ]:
try:
    # Load model pretrained (giả định file wiki.vi.vec nằm cùng thư mục)
    fasttext_model = KeyedVectors.load_word2vec_format(CONFIG["pretrained_path"])
except FileNotFoundError:
    print("Không tìm thấy file, đang dùng model nhỏ thay thế...")
    import gensim.downloader as api
    fasttext_model = api.load("glove-twitter-25") 

# Chuyển đổi dữ liệu văn bản sang vector (sử dụng list comprehension thay cho for tường minh)
X_train = np.array([get_sentence_vector(t, fasttext_model) for t in X_train_raw])
X_test = np.array([get_sentence_vector(t, fasttext_model) for t in X_test_raw])

In [ ]:
# Khởi tạo và huấn luyện mô hình MLP
clf = MLPClassifier(hidden_layer_sizes=CONFIG["hidden_layers"], max_iter=CONFIG["max_iter"])
clf.fit(X_train, y_train)

In [ ]:
# Dự đoán và đánh giá kết quả
y_pred = clf.predict(X_test)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")

# Hiển thị Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()